# Seminar 1: Introduction to LLM Operations

**Practical focus:**  
1. connect Python to an LLM through the **Groq API**;  
2. send and improve a **prompt**;  
3. understand why **LLM evaluation is difficult**.

## What you need before we start

1. **VS Code**
2. **Python**
3. VS Code extensions: **Python** and **Jupyter**
4. A free **GroqCloud account and API key**

Create/manage a Groq API key here:

https://console.groq.com/keys

### API-key rule

Your API key works like a password for programmatic access.

- **Do not share it.**
- **Do not write it directly into a notebook that you will upload or send to someone.**
- In this notebook we use `getpass()`, which hides the key while you type/paste it.

# 1. Install the packages

A Python **package** is reusable code written by somebody else.

Here we need:

- `groq` → communicate with Groq's LLM API;
- `nltk` → calculate BLEU;
- `rouge-score` → calculate ROUGE.

`%pip install` installs the packages into the Python environment used by this notebook.

In [ ]:
%pip install -q groq nltk rouge-score

# 2. Import the tools we need

A few Python basics:

- `#` starts a **comment** — Python ignores it.
- `=` assigns a value to a **variable**.
- Text inside quotation marks is a **string**.
- `print(...)` displays something.
- `from ... import ...` loads a tool from an installed package.

In [ ]:
# getpass lets us type an API key without displaying it on the screen
from getpass import getpass

# Groq is the Python client used to communicate with the Groq API
from groq import Groq

# These tools will be used later for simple text evaluation
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Store the model name in one variable so it is easy to change later
MODEL = "openai/gpt-oss-20b"

print("Model used in this notebook:", MODEL)

# 3. Authenticate with the Groq API

An **API** allows one program to communicate with another service.

In our case:

**Python in VS Code → Groq API → hosted LLM → Groq API → Python**

The API key tells Groq **which account is making the request**.

The key entered below is stored only in the current Python session.

In [ ]:
# getpass() asks for text but hides what you enter
api_key = ""

# Create a Groq client
# We will use this 'client' object to send requests to the API
client = Groq(api_key=api_key)

print("Groq client created.")

Groq client created.


# 4. Send the first prompt

Before running the next cell, identify these three things:

1. **model** → which LLM should Groq run?
2. **prompt/message** → what are we asking?
3. **response** → where does the generated answer come back?

### Python detail

`messages=[{...}]` is a list containing a small dictionary.

You do not need to master lists and dictionaries yet. For now, understand that this structure tells the API:

> this message came from the **user**, and this is its **content**.

In [23]:
# A variable containing our prompt
prompt = "Explain what a large language model is in exactly three short sentences."

# Send the request to Groq
response = client.chat.completions.create(
    model=MODEL,

    # The message sent to the model
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],

    # GPT-OSS is a reasoning model
    # Low effort is enough for this simple task
    reasoning_effort="low",

    # We only want the final answer, not separate reasoning content
    include_reasoning=False,

    # Lower temperature usually gives more stable/less variable answers
    temperature=0.2,

    # Maximum number of tokens available for the generated response
    max_completion_tokens=500
)

# Extract the text from the structured API response
answer = response.choices[0].message.content

print(answer)

A large language model is a type of artificial intelligence that learns patterns in text by training on massive amounts of data. It uses deep neural networks to predict the next word in a sentence, enabling it to generate coherent responses. The model’s size—measured in billions of parameters—allows it to capture nuanced language and knowledge across many domains.


## What just happened?

The important workflow is:

**prompt**  
↓  
**Python sends an API request**  
↓  
**Groq runs the selected LLM**  
↓  
**the API returns a structured response**  
↓  
**Python extracts the generated text**

`response.choices[0].message.content`: The main idea here is that an API response contains several pieces of information, and this expression selects the generated text.

# 5. Create a reusable function

A **function** groups reusable instructions.

Instead of copying the full API request every time, we will create:

```python
ask_llm(prompt)
```

After this cell, we can send a new prompt with only one line of code.

In [24]:
def ask_llm(prompt):
    """Send a prompt to Groq and return the generated text."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        reasoning_effort="low",
        include_reasoning=False,
        temperature=0.2,
        max_completion_tokens=600
    )

    return response.choices[0].message.content

In [25]:
# Calling our function:
answer = ask_llm("Give me three examples of tasks that LLMs can perform.")

print(answer)

Here are three common tasks that large language models (LLMs) can perform:

| # | Task | What it involves | Typical use‑case |
|---|------|------------------|------------------|
| 1 | **Text generation / completion** | The model predicts the next word(s) in a sequence, producing coherent paragraphs, stories, or code snippets. | Writing blog posts, drafting emails, auto‑generating code comments. |
| 2 | **Question answering / information retrieval** | The model interprets a user’s question, searches its internal knowledge base (or external sources), and produces a concise answer. | Customer support bots, FAQ assistants, quick fact lookup. |
| 3 | **Text summarization** | The model condenses a longer document into a shorter, high‑level summary while preserving key points. | Summarizing news articles, meeting minutes, legal documents. |

These tasks illustrate the breadth of LLM capabilities—from creative content creation to practical information extraction.


# 6. Prompt quality: vague vs. structured

Run both prompts and compare the answers.

### Prompt A
Very little guidance.

### Prompt B
Defines:
- the audience,
- what information is required,
- the output format,
- the length.

The **model is the same** in both cases. Only the input changes.

In [26]:
prompt_a = "Tell me about Transformers."

prompt_b = """
Explain the Transformer architecture to Master's students
who know basic machine learning but are new to NLP.

Requirements:
- maximum 100 words
- explain self-attention
- explain why Transformers were important for scaling language models
- do not use equations
- finish with one sentence containing the main takeaway
"""

answer_a = ask_llm(prompt_a)
answer_b = ask_llm(prompt_b)

print("=== VAGUE PROMPT ===")
print(answer_a)

print("\n=== STRUCTURED PROMPT ===")
print(answer_b)

=== VAGUE PROMPT ===
## Transformers – A Quick‑Start Guide

| Section | What you’ll learn |
|---------|-------------------|
| 1️⃣  What is a Transformer? | The core idea: self‑attention + feed‑forward layers. |
| 2️⃣  Why it matters | How it solved the “long‑range dependency” problem in NLP. |
| 3️⃣  Anatomy of a Transformer | Encoder, decoder, multi‑head attention, positional encoding, etc. |
| 4️⃣  Key Variants | BERT, GPT, T5, Vision‑Transformer (ViT), etc. |
| 5️⃣  Training tricks | Masking, learning‑rate schedules, mixed‑precision, etc. |
| 6️⃣  Applications | Language, vision, audio, multimodal, reinforcement learning. |
| 7️⃣  Resources | Papers, codebases, tutorials, datasets. |

---

### 1️⃣  What is a Transformer?

A Transformer is a neural network architecture introduced in the 2017 paper *“Attention Is All You Need”* by Vaswani et al. It replaces recurrent or convolutional layers with **self‑attention** mechanisms, enabling:

- **Parallel training** (no sequential dependenc

## Short discussion

Which answer is **better**? What does **better** mean?

Do not stop at "I prefer this one." Define the criterion.

Possible criteria:

- factual correctness;
- relevance;
- clarity;
- completeness;
- instruction following;
- suitability for the intended audience.

### Key idea

> **A better-specified prompt makes the desired output easier for both the model and the evaluator to understand.**

# 7. Your mini-task — improve a prompt

### Scenario

You want an LLM to explain an environmental topic:

> **How can urban green spaces help cities adapt to extreme heat?**

The first prompt is intentionally vague.

### Your task

Write a better prompt that specifies at least:

1. the intended audience;
2. the required content;
3. an output format;
4. a length constraint.

Then run both and compare the outputs.

In [ ]:
# Vague version
environment_prompt_a = "Explain urban green spaces and heat."

# YOUR TASK:
# Replace TODO with a better structured prompt.
environment_prompt_b = """
TODO
"""

print("=== VAGUE VERSION ===")
print(ask_llm(environment_prompt_a))

# Uncomment after completing your prompt:
# print("\n=== YOUR STRUCTURED VERSION ===")
# print(ask_llm(environment_prompt_b))

# 8. Why do we evaluate LLM outputs?

Generating an answer is only half of an LLM workflow.

We also need to ask:

> **How do we know whether the output is good?**

We will look at three simple reference-based metrics:

- **Exact Match**
- **BLEU**
- **ROUGE**

The purpose is **not** to master these metrics today.

The purpose is to discover their limitations.

In [28]:
# One reference answer
reference = (
    "Large language models generate text by predicting the next token "
    "from the available context."
)

# Exactly the same meaning and wording
candidate_exact = reference

# Correct meaning, different wording
candidate_paraphrase = (
    "LLMs produce text by estimating which token is most likely to come next "
    "given the preceding context."
)

# Factually wrong, but shares several words with the reference
candidate_wrong = (
    "Large language models retrieve a complete answer from a database "
    "of sentences stored during training."
)

# 9. Exact Match

Exact Match asks a very simple question:

> Are the two normalized texts identical?

This can be useful for:
- labels (`POSITIVE`, `NEGATIVE`);
- IDs;
- short answers with one required form.

It is usually too strict for open-ended natural language.

In [29]:
def normalize_text(text):
    # lower() makes everything lowercase
    # strip() removes extra space at the beginning/end
    # split() + join() removes repeated spaces
    return " ".join(text.lower().strip().split())


def exact_match(prediction, reference):
    # int(True) = 1 and int(False) = 0
    return int(normalize_text(prediction) == normalize_text(reference))


print("Exact copy:       ", exact_match(candidate_exact, reference))
print("Correct paraphrase:", exact_match(candidate_paraphrase, reference))
print("Wrong answer:     ", exact_match(candidate_wrong, reference))

Exact copy:        1
Correct paraphrase: 0
Wrong answer:      0


### Question

The correct paraphrase receives **0**.

Does that mean the paraphrase is wrong?

**No.** It only means that it is not textually identical to the reference.

# 10. BLEU and ROUGE

You do **not** need to understand every line of the metric implementation.

For today, remember:

- **BLEU** looks at overlapping word sequences (**n-grams**);
- **ROUGE** also measures overlap with a reference;

Neither one directly checks whether a statement is factually true!

In [ ]:
# ----- BLEU -----

def bleu_score(prediction, reference):
    prediction_words = normalize_text(prediction).split()
    reference_words = normalize_text(reference).split()

    return sentence_bleu(
        [reference_words],
        prediction_words,
        smoothing_function=SmoothingFunction().method1
    )


# ----- ROUGE-L -----

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l(prediction, reference):
    score = rouge.score(reference, prediction)
    return score["rougeL"].fmeasure


# Compare a correct paraphrase and a wrong answer
print("CORRECT PARAPHRASE")
print("BLEU:   ", round(bleu_score(candidate_paraphrase, reference), 3))
print("ROUGE-L:", round(rouge_l(candidate_paraphrase, reference), 3))

print("\nWRONG ANSWER")
print("BLEU:   ", round(bleu_score(candidate_wrong, reference), 3))
print("ROUGE-L:", round(rouge_l(candidate_wrong, reference), 3))

# 11. The key evaluation lesson

You may observe that the **factually wrong answer receives a higher BLEU score** than the correct paraphrase.

Why?

Because BLEU measures **word / n-gram overlap**, not factual truth.

### Main takeaway

> **A high score is useful only if the metric measures the property we actually care about.**

Therefore:

- Exact Match → useful for strict outputs;
- BLEU / ROUGE → useful for forms of reference similarity;
- factual correctness → usually needs other evaluation methods;
- human judgment is still important for many open-ended tasks.

# Final 5-minute discussion

Try to answer these four questions without looking back:

1. What is the role of the **Groq API**?
2. Why do we need an **API key**?
3. Why can a structured prompt produce a better output without changing the model?
4. Why can a wrong answer receive a higher BLEU/ROUGE score than a correct paraphrase?

### Seminar 1 takeaway

**LLM operation = sending well-defined inputs to a model through an API and critically evaluating the outputs.**

# OPTIONAL

Try a simple hallucination / false-premise test:

> "Summarize the main contribution of the Vienna Transformer Protocol, introduced by Vaswani and Devlin in 2019."

Then improve the prompt so that the model checks the premise and avoids inventing unsupported details.

In [33]:
# OPTIONAL

false_premise_prompt = """
Summarize the main contribution of the Vienna Transformer Protocol,
introduced by Vaswani and Devlin in 2019.
Do not invent any new information, and do not hallucinate any details.
Only provide information that is verifiable and based on established sources.
"""

print(ask_llm(false_premise_prompt))

I’m not aware of any publication or protocol titled the “Vienna Transformer Protocol” by Vaswani and Devlin from 2019. The most prominent work by these authors in that period is the 2017 paper *Attention Is All You Need* (Vaswani et al.) that introduced the Transformer architecture, and later works by Devlin (e.g., BERT) that build on that architecture. No established source documents a separate “Vienna Transformer Protocol” from 2019, so I cannot summarize a contribution that does not appear in the literature.
